# Chapter 66: Incident Response Automation

**Volume 4 — Security Operations**

**The attacker was in your environment for 197 days.** You had 50,000 alerts.
Your team triaged 200. The breach cost $4.45M. The gap between 197 days and 24 hours
is not talent — it is **architecture**. Automated playbooks execute in seconds.
Manual response takes hours. This chapter builds the pipeline that changes that math.

### What You Will Build — 5 IR Automation Demos

| Demo | Technique | What It Does |
|------|-----------|-------------|
| **1. Confidence-Gated Playbook Executor** | Risk-tier thresholds | Runs low-risk actions autonomously, defers high-risk to humans |
| **2. Checkpoint and Rollback Engine** | Immutable action log + reversal | Undoes every automated action on false positive confirmation |
| **3. Actor-Critic Safety Architecture** | Dual-agent proposer + validator | Blocks unsafe actions before they execute |
| **4. Evidence Preservation with Chain of Custody** | SHA-256 hashing + immutable log | Collects and seals forensic artifacts before containment |
| **5. Full IR Pipeline with MTTD/MTTR Metrics** | End-to-end orchestration + LLM brief | Measures detection and response time, generates incident brief |

### The Core Insight
> **Manual incident response operates at human speed: analysts read, escalate, gather, decide, act — hours to days.
> Automated IR operates at machine speed: detection fires, playbook executes, containment runs — minutes.
> The gap between those speeds is the gap between a $1.3M incident and a $4.45M breach.
> Speed is the only variable that matters, and speed requires architecture.**

In [ ]:
# pip install anthropic
# Incident Response Automation — pure Python, no ML libraries required!

import os, json, re, math, time, random, hashlib
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from collections import defaultdict

# ── Anthropic client ──────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("LLM analyst: ACTIVE (Anthropic API)")
else:
    print("LLM analyst: SIMULATION MODE (set ANTHROPIC_API_KEY for real analysis)")

def llm_analyze(prompt: str, max_tokens: int = 200) -> str:
    if USE_API:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.content[0].text.strip()
    # Simulation fallback — domain-specific canned responses
    if "ransomware" in prompt.lower():
        return ("CONFIRMED: Ransomware - T1486 Data Encrypted for Impact. "
                "Priorities: 1) Isolate endpoint immediately 2) Disable account "
                "3) Snapshot memory before reboot. Evidence: volume shadow copy deletion confirms encryption intent.")
    if "credential" in prompt.lower() or "lateral" in prompt.lower():
        return ("CONFIRMED: Credential compromise / lateral movement - T1078 Valid Accounts. "
                "Priorities: 1) Revoke sessions 2) Disable account 3) Audit all access since first anomaly. "
                "Evidence: impossible travel + new host access confirms compromise.")
    if "false positive" in prompt.lower() or "rollback" in prompt.lower():
        return "FALSE POSITIVE confirmed. All automated actions have been reversed. No further action required."
    return "Simulation: " + prompt[:80] + "..."

print("Setup complete. All 5 IR automation demos ready.")

## Demo 1: Confidence-Gated Playbook Executor

A **response playbook** is the IR equivalent of a network runbook: exact steps, exact order,
exact rollback. The difference is execution speed — seconds via API instead of minutes via CLI.

The core insight is **confidence gating** — every action has a risk tier, and every risk tier
has a minimum confidence threshold before autonomous execution is permitted:

| Risk Tier | Confidence Threshold | Action Examples |
|-----------|---------------------|----------------|
| **Low** | ≥ 0.70 | Collect evidence, add to watchlist, revoke sessions |
| **Medium** | ≥ 0.85 | Disable account, restrict host, block IP at firewall |
| **High** | ≥ 0.95 | Isolate endpoint, delete file, reconfigure VLAN |

Below the threshold for a given tier: the action is **deferred** — placed in the human
approval queue with full context. Above threshold: autonomous execution with rollback logged.

*Network analogy: confidence gating is like a staged BGP route policy.
Prefixes with full RPKI validation pass automatically. Prefixes with ROA mismatch
go to the manual review queue. Same threshold logic, different domain.*

In [ ]:
# ── Demo 1: Confidence-Gated Playbook Executor ────────────────────────────────

@dataclass
class Action:
    """A containment action with risk tier and rollback procedure."""
    name: str
    description: str
    rollback: str        # API call to reverse; 'no_rollback_needed' for read-only
    risk: str            # low | medium | high
    api_endpoint: str    # simulated API target

@dataclass
class Incident:
    id: str
    type: str            # ransomware | credential_compromise | lateral_movement
    host: str
    user: str
    confidence: float    # 0.0 - 1.0
    evidence: List[str]
    actions_taken: List[str] = field(default_factory=list)
    detected_at: float = field(default_factory=time.time)

# Playbook library — ordered by execution priority
PLAYBOOKS: Dict[str, List[Action]] = {
    "ransomware": [
        Action("isolate_endpoint",
               "Block all network traffic from host (EDR API — exceptions: IR VLAN)",
               "lift_isolation(host_id)",
               "high",
               "POST /edr/v1/hosts/{host}/isolate"),
        Action("disable_account",
               "Suspend user account in identity provider — force re-auth on lift",
               "enable_account(user_id)",
               "medium",
               "PATCH /idp/v1/users/{user}/disable"),
        Action("collect_evidence",
               "Snapshot process list, open files, network sockets (volatile forensics)",
               "no_rollback_needed",
               "low",
               "POST /edr/v1/forensics/{host}/snapshot"),
        Action("block_c2_ips",
               "Create deny rule at perimeter firewall for known C2 infrastructure",
               "remove_firewall_rule(rule_id)",
               "low",
               "POST /fw/v1/policy/rules"),
    ],
    "credential_compromise": [
        Action("collect_evidence",
               "Capture session list, auth history, access log since first anomaly",
               "no_rollback_needed",
               "low",
               "GET /siem/v1/auth-events?user={user}&hours=48"),
        Action("revoke_sessions",
               "Invalidate all active OAuth/SAML sessions for this account",
               "no_rollback_needed",
               "low",
               "DELETE /idp/v1/users/{user}/sessions"),
        Action("disable_account",
               "Suspend account pending credential rotation and MFA re-enrollment",
               "enable_account(user_id)",
               "medium",
               "PATCH /idp/v1/users/{user}/disable"),
        Action("add_to_watchlist",
               "Flag all hosts accessed in this session for enhanced monitoring",
               "remove_from_watchlist(host_ids)",
               "low",
               "POST /siem/v1/watchlist/hosts"),
    ],
    "lateral_movement": [
        Action("collect_evidence",
               "Snapshot auth logs and NetFlow from source host (last 2 hours)",
               "no_rollback_needed",
               "low",
               "GET /siem/v1/netflow?src={host}&hours=2"),
        Action("restrict_source_host",
               "Apply ACL blocking new RDP/SMB connections from source host",
               "remove_acl_restriction(host_id)",
               "medium",
               "POST /fw/v1/acl/host/{host}/restrict"),
        Action("flag_touched_hosts",
               "Add all hosts the attacker reached to high-priority watchlist",
               "remove_from_watchlist(host_ids)",
               "low",
               "POST /siem/v1/watchlist/hosts"),
        Action("isolate_source_host",
               "Full network isolation of the lateral movement source host",
               "lift_isolation(host_id)",
               "high",
               "POST /edr/v1/hosts/{host}/isolate"),
    ],
}

# Confidence thresholds per risk tier
THRESHOLDS = {"low": 0.70, "medium": 0.85, "high": 0.95}


def execute_playbook(incident: Incident) -> dict:
    """
    Execute containment playbook with confidence gating.
    Returns: executed actions, deferred (needs human), rollback log.
    """
    playbook = PLAYBOOKS.get(incident.type, [])
    executed   = []
    deferred   = []
    rollback_log = []

    print(f"\n[PLAYBOOK] {incident.id} — {incident.type.upper().replace('_', ' ')}")
    print(f"  Confidence: {incident.confidence:.0%} | Host: {incident.host} | User: {incident.user}")
    print(f"  Playbook: {len(playbook)} actions defined")
    print()

    for action in playbook:
        threshold = THRESHOLDS[action.risk]

        if incident.confidence >= threshold:
            # Meets threshold — autonomous execution
            executed.append(action.name)
            incident.actions_taken.append(action.name)
            if action.rollback != "no_rollback_needed":
                rollback_log.append({
                    "action": action.name,
                    "rollback_call": action.rollback,
                    "api_endpoint": action.api_endpoint,
                    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                    "incident_id": incident.id,
                })
            status = "AUTO-EXECUTED"
            gate   = f"conf {incident.confidence:.0%} >= threshold {threshold:.0%}"
            print(f"  [OK] {status}: {action.name}")
            print(f"       {action.description}")
            print(f"       API: {action.api_endpoint} | Gate: {gate}")
        else:
            # Below threshold — defer to human
            deferred.append({
                "action": action.name,
                "description": action.description,
                "risk": action.risk,
                "reason": f"conf {incident.confidence:.0%} < threshold {threshold:.0%} for {action.risk}-risk",
                "api_endpoint": action.api_endpoint,
            })
            print(f"  [DEFER] HUMAN REQUIRED: {action.name} ({action.risk}-risk)")
            print(f"          Need {threshold:.0%}, have {incident.confidence:.0%}")
        print()

    return {"executed": executed, "deferred": deferred, "rollback_log": rollback_log}


# ── Test with three incidents at different confidence levels ──────────────────
test_incidents = [
    Incident(
        id="INC-2026-0412",
        type="credential_compromise",
        host="workstation-eng-07",
        user="j.smith",
        confidence=0.88,
        evidence=[
            "Login from 185.220.101.47 (Tor exit node) at 03:14 UTC",
            "Previous login from 10.0.1.22 (internal) 90 min ago — impossible travel",
            "MFA not presented (RADIUS: password-only auth)",
            "Accessed db-prod-01 — first access to this host in 90 days",
        ],
    ),
    Incident(
        id="INC-2026-0413",
        type="ransomware",
        host="file-server-03",
        user="svc-backup",
        confidence=0.72,  # below medium and high thresholds
        evidence=[
            "vssadmin.exe delete shadows /all executed at 02:47 UTC",
            "1,200 file rename operations in 90 seconds (.docx -> .encrypted)",
            "Ransom note README.txt dropped in C:\\Users\\Public",
        ],
    ),
]

print("=" * 65)
print("CONFIDENCE-GATED PLAYBOOK EXECUTOR")
print("=" * 65)

all_results = []
for inc in test_incidents:
    result = execute_playbook(inc)
    all_results.append((inc, result))
    print(f"  SUMMARY for {inc.id}:")
    print(f"    Executed autonomously : {result['executed']}")
    print(f"    Deferred (human req.) : {[d['action'] for d in result['deferred']]}")
    print(f"    Rollback entries      : {len(result['rollback_log'])} reversible actions logged")
    print()

print("[PLAYBOOK] Deferred actions routed to analyst approval queue.")

## Demo 2: Checkpoint and Rollback Engine

Every automated IR system will eventually act on a false positive.
The question is not whether — it is whether your system can **undo the damage in seconds**.

**Checkpoint-and-rollback** borrows from transactional database systems:
before executing any action, record the pre-action state (the checkpoint).
If the action proves incorrect, revert the checkpoint.

The implementation uses an **immutable action log** — append-only, never overwritten:

```
Automated actions executed:
  T+00s  collect_evidence      (no rollback needed)
  T+02s  revoke_sessions       (no rollback needed)
  T+04s  disable_account       -> rollback: enable_account(j.smith)
  T+06s  add_to_watchlist      -> rollback: remove_from_watchlist([host1, host2])

Analyst marks as FALSE POSITIVE at T+12min:
  T+12m  ROLLBACK add_to_watchlist   (reverse chronological)
  T+12m  ROLLBACK disable_account    (account re-enabled)
```

**Rollback must be automated.** A false positive that isolates the wrong server and
requires manual reversal is asymmetric: harm at machine speed, repair at human speed.

*Network analogy: rollback is Git's commit history for your security actions.
You can always go back to any prior commit. Without it, every automated action
is a one-way door.*

In [ ]:
# ── Demo 2: Checkpoint and Rollback Engine ────────────────────────────────────

@dataclass
class ActionRecord:
    """Immutable record of one executed action — never modified after creation."""
    incident_id: str
    sequence_num: int
    action_name: str
    api_endpoint: str
    parameters: Dict
    rollback_call: str         # 'no_rollback_needed' if read-only
    pre_state_hash: str        # SHA-256 of system state before action
    executed_at: str
    rolled_back: bool = False
    rolled_back_at: str = ""


class ActionLog:
    """
    Append-only action log — the source of truth for all automated actions.
    Supports checkpoint capture and full incident rollback.
    In production: backed by Kafka / append-only database (no UPDATE/DELETE).
    """

    def __init__(self):
        self._records: List[ActionRecord] = []  # append-only
        self._sequence: int = 0

    def _capture_state_hash(self, incident_id: str, action_name: str) -> str:
        """Snapshot current system state as SHA-256 hash (simulated)."""
        state_repr = f"{incident_id}:{action_name}:{time.time()}"
        return hashlib.sha256(state_repr.encode()).hexdigest()[:16]

    def record(self, incident_id: str, action_name: str,
               api_endpoint: str, parameters: Dict,
               rollback_call: str) -> ActionRecord:
        """Append a new action record — called before executing the action."""
        self._sequence += 1
        rec = ActionRecord(
            incident_id=incident_id,
            sequence_num=self._sequence,
            action_name=action_name,
            api_endpoint=api_endpoint,
            parameters=parameters,
            rollback_call=rollback_call,
            pre_state_hash=self._capture_state_hash(incident_id, action_name),
            executed_at=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        )
        self._records.append(rec)
        return rec

    def get_incident_records(self, incident_id: str) -> List[ActionRecord]:
        """All records for a given incident, in execution order."""
        return [r for r in self._records if r.incident_id == incident_id]

    def rollback_incident(self, incident_id: str) -> List[str]:
        """
        Reverse all reversible actions for an incident in reverse chronological order.
        Returns list of rollback calls executed.
        In production: calls the actual reversal API endpoints.
        """
        records = self.get_incident_records(incident_id)
        reversible = [
            r for r in records
            if r.rollback_call != "no_rollback_needed" and not r.rolled_back
        ]
        # Reverse chronological — undo in opposite order to execution
        reversible_sorted = sorted(reversible, key=lambda r: r.sequence_num, reverse=True)

        rollbacks_executed = []
        now_str = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

        for rec in reversible_sorted:
            # In production: call the rollback API here
            rec.rolled_back = True
            rec.rolled_back_at = now_str
            rollbacks_executed.append(rec.rollback_call)
        return rollbacks_executed

    def print_log(self, incident_id: str):
        """Print the full action log for an incident."""
        records = self.get_incident_records(incident_id)
        print(f"\n  ACTION LOG — {incident_id} ({len(records)} entries)")
        print(f"  {'#':<4} {'Action':<28} {'State Hash':<18} {'Rolled Back'}")
        print("  " + "-" * 62)
        for r in records:
            rb_str = f"YES @ {r.rolled_back_at}" if r.rolled_back else "no"
            print(f"  {r.sequence_num:<4} {r.action_name:<28} {r.pre_state_hash:<18} {rb_str}")


# ── Simulate a credential compromise response with rollback ───────────────────
print("=" * 65)
print("CHECKPOINT AND ROLLBACK ENGINE")
print("=" * 65)

action_log = ActionLog()
incident_id = "INC-2026-0415"

# Simulate executing playbook and recording each action
planned_actions = [
    ("collect_evidence",   "GET /siem/v1/auth-events?user=m.garcia&hours=48",
     {"user": "m.garcia", "hours": 48}, "no_rollback_needed"),
    ("revoke_sessions",    "DELETE /idp/v1/users/m.garcia/sessions",
     {"user": "m.garcia"}, "no_rollback_needed"),
    ("disable_account",   "PATCH /idp/v1/users/m.garcia/disable",
     {"user": "m.garcia", "reason": incident_id}, "enable_account(m.garcia)"),
    ("add_to_watchlist",   "POST /siem/v1/watchlist/hosts",
     {"hosts": ["db-prod-01", "file-srv-02"]}, "remove_from_watchlist([db-prod-01,file-srv-02])"),
]

print(f"\nPhase 1: Executing playbook for {incident_id}")
print("-" * 50)
for name, endpoint, params, rollback in planned_actions:
    rec = action_log.record(incident_id, name, endpoint, params, rollback)
    rb_note = "(reversible)" if rollback != "no_rollback_needed" else "(read-only, no rollback)"
    print(f"  [SEQ {rec.sequence_num}] Executed: {name} {rb_note}")
    print(f"           State checkpoint: {rec.pre_state_hash}")

# Print the full log
action_log.print_log(incident_id)

# Analyst review — marks it a false positive
print(f"\nPhase 2: Analyst review (simulated 12-minute window)")
print("-" * 50)
analyst_verdict = "false_positive"  # change to 'confirmed' in real scenario
print(f"  Analyst verdict: {analyst_verdict.upper()}")

if analyst_verdict == "false_positive":
    llm_note = llm_analyze(
        f"Analyst marked {incident_id} as false positive. Initiating rollback. "
        f"What should the analyst communicate to the affected user? Under 60 words.",
        max_tokens=80
    )
    print(f"  LLM communication draft: {llm_note}")

    print(f"\nPhase 3: Automated rollback — reverse chronological order")
    print("-" * 50)
    rollbacks = action_log.rollback_incident(incident_id)
    for i, rb in enumerate(rollbacks, 1):
        print(f"  [ROLLBACK {i}] Executing: {rb}")

    action_log.print_log(incident_id)
    print(f"\n  Rollback complete. {len(rollbacks)} actions reversed.")
    print("  Time from detection to rollback: seconds (vs hours for manual reversal)")

## Demo 3: Actor-Critic Safety Architecture

The **actor-critic pattern** is the fundamental safety mechanism for autonomous IR.
No single agent should propose AND approve its own actions.

```
Alert fires
    |
    v
[ACTOR AGENT]  — analyzes incident, proposes response action
    |
    v
[CRITIC AGENT] — independently evaluates proposal against safety policy
    |              Checks: critical asset? blast radius? reversible? threshold met?
    |
   [APPROVE]        [REJECT + ESCALATE TO HUMAN]
    |                    |
    v                    v
 Execute             Human queue
 + log               + actor proposal
                      + critic objections
```

The critic checks five conditions before approving any action:

| Check | Logic | Block if... |
|-------|-------|------------|
| Critical asset | Is target in the protected list? | Target is production database or core infrastructure |
| Blast radius | How many users/services affected? | > 50 users or customer-facing service |
| Reversibility | Is there a rollback procedure? | Action is permanent or destructive |
| Confidence gate | Does score meet risk-tier threshold? | Confidence below required threshold |
| Playbook alignment | Is action in the approved playbook? | Actor proposes action not in playbook |

*Network analogy: actor-critic is emergency change management at machine speed.
The on-call engineer (actor) proposes the fix. The senior reviewer (critic) checks:
"Does this affect OSPF to site B? Is there a rollback?" Same discipline, different speed.*

In [ ]:
# ── Demo 3: Actor-Critic Safety Architecture ──────────────────────────────────

# Safety policy: assets where autonomous action is prohibited
CRITICAL_ASSETS = {
    "db-prod-01", "db-prod-02",   # production databases
    "dc-01", "dc-02",             # domain controllers
    "payment-gw-01",              # payment gateway
    "core-fw-01", "core-fw-02",   # core firewalls
}

@dataclass
class ProposedAction:
    """An action proposed by the actor agent — awaiting critic evaluation."""
    action_name: str
    target_host: str
    target_user: str
    risk_tier: str          # low | medium | high
    confidence: float
    reversible: bool
    rollback_call: str
    affected_users_estimate: int
    customer_facing: bool
    in_playbook: bool
    actor_rationale: str

@dataclass
class CriticVerdict:
    approved: bool
    checks_passed: List[str]
    checks_failed: List[str]
    objections: List[str]
    escalation_note: str = ""


def actor_propose(incident: Incident, action: Action) -> ProposedAction:
    """
    Actor agent: analyze incident and formulate a proposed response action.
    In production: LLM-generated proposal with full context.
    """
    prompt = (
        f"IR incident {incident.id}: type={incident.type}, host={incident.host}, "
        f"user={incident.user}, confidence={incident.confidence:.0%}. "
        f"Evidence: {'; '.join(incident.evidence[:2])}. "
        f"I am proposing action: {action.name} ({action.description}). "
        f"Provide a one-sentence rationale for this action. Under 40 words."
    )
    rationale = llm_analyze(prompt, max_tokens=60)

    return ProposedAction(
        action_name=action.name,
        target_host=incident.host,
        target_user=incident.user,
        risk_tier=action.risk,
        confidence=incident.confidence,
        reversible=(action.rollback != "no_rollback_needed"),
        rollback_call=action.rollback,
        affected_users_estimate=random.randint(1, 3),  # in production: from CMDB
        customer_facing=(incident.host in {"payment-gw-01", "web-prod-01"}),
        in_playbook=True,
        actor_rationale=rationale,
    )


def critic_evaluate(proposal: ProposedAction) -> CriticVerdict:
    """Critic agent: independently evaluate proposed action against safety policy."""
    checks_passed = []
    checks_failed = []
    objections = []

    # Check 1: Critical asset protection
    if proposal.target_host in CRITICAL_ASSETS:
        checks_failed.append("critical_asset")
        objections.append(
            f"Target {proposal.target_host} is in critical asset list — "
            f"autonomous action prohibited, requires senior analyst approval"
        )
    else:
        checks_passed.append("critical_asset")

    # Check 2: Blast radius
    if proposal.affected_users_estimate > 50 or proposal.customer_facing:
        checks_failed.append("blast_radius")
        objections.append(
            f"Blast radius too large: {proposal.affected_users_estimate} users affected "
            f"or customer-facing service — requires CAB approval"
        )
    else:
        checks_passed.append("blast_radius")

    # Check 3: Reversibility (high-risk must be reversible)
    if proposal.risk_tier == "high" and not proposal.reversible:
        checks_failed.append("reversibility")
        objections.append(
            f"High-risk action with no rollback procedure — cannot auto-execute"
        )
    else:
        checks_passed.append("reversibility")

    # Check 4: Confidence gate
    threshold = THRESHOLDS[proposal.risk_tier]
    if proposal.confidence < threshold:
        checks_failed.append("confidence_gate")
        objections.append(
            f"Confidence {proposal.confidence:.0%} below threshold {threshold:.0%} "
            f"for {proposal.risk_tier}-risk action"
        )
    else:
        checks_passed.append("confidence_gate")

    # Check 5: Playbook alignment
    if not proposal.in_playbook:
        checks_failed.append("playbook_alignment")
        objections.append("Proposed action not in approved playbook — off-playbook actions require human decision")
    else:
        checks_passed.append("playbook_alignment")

    approved = len(checks_failed) == 0
    escalation = ""
    if not approved:
        escalation = (
            f"Action '{proposal.action_name}' blocked by critic. "
            f"Escalating to human queue with {len(objections)} objection(s)."
        )

    return CriticVerdict(
        approved=approved,
        checks_passed=checks_passed,
        checks_failed=checks_failed,
        objections=objections,
        escalation_note=escalation,
    )


# ── Run actor-critic on test scenarios ────────────────────────────────────────
print("=" * 65)
print("ACTOR-CRITIC SAFETY ARCHITECTURE")
print("=" * 65)

test_scenario = Incident(
    id="INC-2026-0416",
    type="lateral_movement",
    host="workstation-dev-12",
    user="t.nguyen",
    confidence=0.91,
    evidence=[
        "RDP connection from workstation-dev-12 to db-prod-01 at 01:33 UTC",
        "t.nguyen has no historical access to db-prod-01 in 180-day baseline",
        "5 new host connections in 22 minutes (baseline: 0.3/hour)",
    ],
)

# Actor proposes each playbook action, critic evaluates
lateral_playbook = PLAYBOOKS["lateral_movement"]

for action in lateral_playbook:
    proposal = actor_propose(test_scenario, action)
    verdict  = critic_evaluate(proposal)

    print(f"\n[ACTOR] Proposes: {proposal.action_name} ({proposal.risk_tier}-risk)")
    print(f"  Rationale: {proposal.actor_rationale}")
    print(f"  Target: {proposal.target_host} | Confidence: {proposal.confidence:.0%}")
    print(f"  Reversible: {proposal.reversible} | In playbook: {proposal.in_playbook}")

    print(f"[CRITIC] Verdict: {'APPROVED' if verdict.approved else 'BLOCKED'}")
    print(f"  Passed checks: {verdict.checks_passed}")
    if verdict.checks_failed:
        print(f"  Failed checks: {verdict.checks_failed}")
        for obj in verdict.objections:
            print(f"  Objection: {obj}")
        print(f"  -> {verdict.escalation_note}")
    else:
        print(f"  -> Action cleared for autonomous execution")

print("\n[ACTOR-CRITIC] Safety gate enforced. Blocked actions escalated to human queue.")

## Demo 4: Evidence Preservation with Chain of Custody

**Evidence collection must precede containment.** This sounds obvious.
It is violated constantly: containment reboots the endpoint (destroying memory),
or network isolation blocks the forensic collection server.

The **chain of custody** is the legal and operational proof that evidence
has not been modified since collection. For automated IR, that means:

```
Step 1: Snapshot volatile state (process list, network sockets, loaded modules)
         -> HASH immediately (SHA-256)
         -> Transfer to write-once forensic store
         -> Log: timestamp + hash + collector identity

Step 2: Hash every static artifact at collection time
         -> The hash is the tamper-proof fingerprint

Step 3: ONLY THEN — execute containment actions
         -> Evidence is already sealed, containment cannot destroy it
```

The **immutable custody log** records every access to every artifact:
who touched it, when, what they did. This is what makes the evidence
admissible in legal proceedings AND trustworthy for root-cause analysis.

| Artifact Type | Volatile? | Collection Priority | Destroyed By |
|--------------|-----------|--------------------|--------------|
| Process list | Yes | Immediate (pre-isolation) | Reboot or isolation |
| Network sockets | Yes | Immediate | Isolation |
| Memory dump | Yes | High (pre-reboot) | Reboot |
| Disk image | No | After volatile collection | Malware removal |
| Log files | No | Continuous | Log rotation |

In [ ]:
# ── Demo 4: Evidence Preservation with Chain of Custody ───────────────────────

@dataclass
class ForensicArtifact:
    """A single collected piece of forensic evidence."""
    artifact_id: str
    incident_id: str
    artifact_type: str       # process_list | network_sockets | memory_dump | log_file | disk_image
    source_host: str
    collected_at: str
    collector: str           # automated_ir_system | analyst_workstation
    content_hash: str        # SHA-256 of artifact content
    size_bytes: int
    volatile: bool           # True = would be lost on reboot
    storage_path: str        # write-once forensic store path

@dataclass
class CustodyEvent:
    """One entry in the chain of custody log — immutable."""
    event_id: str
    artifact_id: str
    action: str              # collected | transferred | accessed | analyzed | exported
    actor: str               # system or analyst ID
    timestamp: str
    hash_before: str         # hash of artifact before this action
    hash_after: str          # hash after — must match 'before' for read-only actions
    notes: str = ""


class ForensicStore:
    """
    Write-once forensic evidence store with chain of custody tracking.
    In production: backed by S3 Object Lock (WORM) or similar immutable storage.
    """

    def __init__(self, incident_id: str):
        self.incident_id = incident_id
        self._artifacts: Dict[str, ForensicArtifact] = {}
        self._custody_log: List[CustodyEvent] = []
        self._event_counter = 0

    def _artifact_id(self, artifact_type: str) -> str:
        ts = int(time.time() * 1000) % 100000
        return f"{self.incident_id}-{artifact_type[:6].upper()}-{ts}"

    def _event_id(self) -> str:
        self._event_counter += 1
        return f"CUST-{self._event_counter:04d}"

    def _simulate_hash(self, content_description: str) -> str:
        """Simulate SHA-256 of artifact content."""
        return hashlib.sha256(
            f"{content_description}:{time.time()}".encode()
        ).hexdigest()

    def collect_artifact(
        self,
        artifact_type: str,
        source_host: str,
        content_description: str,
        size_bytes: int,
        volatile: bool,
        collector: str = "automated_ir_system",
    ) -> ForensicArtifact:
        """Collect and seal a forensic artifact — hash at collection time."""
        art_id = self._artifact_id(artifact_type)
        content_hash = self._simulate_hash(content_description)
        storage_path = f"s3://forensics-immutable/{self.incident_id}/{art_id}.bin"
        timestamp = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

        artifact = ForensicArtifact(
            artifact_id=art_id,
            incident_id=self.incident_id,
            artifact_type=artifact_type,
            source_host=source_host,
            collected_at=timestamp,
            collector=collector,
            content_hash=content_hash,
            size_bytes=size_bytes,
            volatile=volatile,
            storage_path=storage_path,
        )
        self._artifacts[art_id] = artifact

        # Record custody event
        self._custody_log.append(CustodyEvent(
            event_id=self._event_id(),
            artifact_id=art_id,
            action="collected",
            actor=collector,
            timestamp=timestamp,
            hash_before="",  # no prior state
            hash_after=content_hash,
            notes=f"Initial collection from {source_host} — volatile={volatile}",
        ))
        return artifact

    def access_artifact(self, artifact_id: str, actor: str, purpose: str):
        """Log an analyst accessing an artifact — hash verified to detect tampering."""
        artifact = self._artifacts.get(artifact_id)
        if not artifact:
            print(f"  [ERROR] Artifact {artifact_id} not found in store")
            return
        self._custody_log.append(CustodyEvent(
            event_id=self._event_id(),
            artifact_id=artifact_id,
            action="accessed",
            actor=actor,
            timestamp=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            hash_before=artifact.content_hash,
            hash_after=artifact.content_hash,  # read-only: must match
            notes=purpose,
        ))

    def print_custody_report(self):
        """Print the full chain of custody report."""
        print(f"\n  CHAIN OF CUSTODY REPORT — {self.incident_id}")
        print(f"  Artifacts collected: {len(self._artifacts)}")
        print(f"  Custody events logged: {len(self._custody_log)}")
        print()
        print(f"  {'Artifact ID':<35} {'Type':<18} {'Size':>8} {'Volatile'} {'Hash (first 16)'}")
        print("  " + "-" * 85)
        for art in self._artifacts.values():
            print(f"  {art.artifact_id:<35} {art.artifact_type:<18} "
                  f"{art.size_bytes:>8,} {str(art.volatile):<9} {art.content_hash[:16]}")
        print()
        print(f"  {'Event ID':<12} {'Action':<12} {'Actor':<28} {'Artifact (short)'}")
        print("  " + "-" * 75)
        for ev in self._custody_log:
            art_short = ev.artifact_id[-12:]
            print(f"  {ev.event_id:<12} {ev.action:<12} {ev.actor:<28} ...{art_short}")


# ── Run evidence collection for a ransomware incident ─────────────────────────
print("=" * 65)
print("EVIDENCE PRESERVATION WITH CHAIN OF CUSTODY")
print("=" * 65)

store = ForensicStore("INC-2026-0417")
host  = "file-server-03"

print(f"\nPhase 1: Volatile evidence collection (BEFORE containment)")
print("-" * 55)

# Volatile artifacts — must collect FIRST
volatile_artifacts = [
    ("process_list",     f"272 processes on {host} at detection time",     48_200, True),
    ("network_sockets",  f"Active TCP/UDP connections snapshot from {host}", 12_800, True),
    ("loaded_modules",   f"DLL/kernel module list from {host}",              8_400,  True),
]

collected_arts = []
for art_type, content_desc, size, volatile in volatile_artifacts:
    art = store.collect_artifact(art_type, host, content_desc, size, volatile)
    collected_arts.append(art)
    vol_label = "VOLATILE" if volatile else "stable"
    print(f"  Collected [{vol_label}]: {art_type}")
    print(f"    Artifact ID : {art.artifact_id}")
    print(f"    SHA-256     : {art.content_hash[:32]}...")
    print(f"    Storage     : {art.storage_path}")
    print()

print("Phase 2: Non-volatile collection (can survive containment)")
print("-" * 55)

stable_artifacts = [
    ("log_file",  f"Windows Security Event Log from {host}", 2_847_000, False),
    ("disk_image", f"Disk image snapshot of C:\\ from {host}", 85_000_000_000, False),
]

for art_type, content_desc, size, volatile in stable_artifacts:
    art = store.collect_artifact(art_type, host, content_desc, size, volatile)
    print(f"  Collected [stable]: {art_type} — {size:,} bytes")

print()
print("Phase 3: Analyst access logged in chain of custody")
print("-" * 55)
# Simulate analyst accessing the process list
store.access_artifact(
    collected_arts[0].artifact_id,
    actor="analyst.chen@company.com",
    purpose="Initial triage — identifying malicious process tree"
)
print(f"  Analyst access logged and hash-verified")

print()
print("Phase 4: NOW safe to execute containment actions")
print("-" * 55)
print("  Containment: isolate_endpoint (volatile evidence already sealed)")
print("  Containment: disable_account  (auth logs already captured)")

store.print_custody_report()
print("\n  All artifacts hash-sealed before containment. Evidence integrity assured.")

## Demo 5: Full IR Pipeline with MTTD/MTTR Metrics

The two metrics that define IR automation value:

**MTTD (Mean Time to Detect)**: from attacker first access to confirmed alert.
- Industry average (no AI): 197 days
- With AI detection (Chapter 70 architecture): < 24 hours

**MTTR (Mean Time to Respond)**: from confirmed alert to fully contained incident.
- Manual response: 2–8 hours (well-understood incidents), days (complex)
- Automated playbook: 1–10 minutes (high-confidence scenarios)

**False Positive Rate**: % of automated actions reversed because detection was wrong.
This metric gates autonomy expansion:

| Risk Tier | Max Acceptable FP Rate | Gate Consequence |
|-----------|------------------------|------------------|
| Low-risk actions | 15% | Exceed: pause auto-execution for this category |
| Medium-risk actions | 5% | Exceed: revert to human approval for this tier |
| High-risk actions | 2% | Exceed: full autonomy suspended |

This demo runs a complete IR pipeline: detection → evidence → playbook → rollback option
→ LLM incident brief → MTTD/MTTR measurement.

> **Non-negotiable guardrail: LLM recommendations require human approval.
> The AI generates the incident brief and recommends next steps.
> The analyst makes the final containment decision.
> Autonomous execution only applies to pre-approved, confidence-gated playbook actions.**

In [ ]:
# ── Demo 5: Full IR Pipeline with MTTD/MTTR Metrics ──────────────────────────

@dataclass
class IRMetrics:
    """MTTD and MTTR tracking for one incident."""
    incident_id: str
    attacker_first_access_ts: float    # when attacker first gained access
    detection_ts: float                # when alert fired
    response_start_ts: float           # when playbook started
    containment_complete_ts: float     # when all actions executed

    @property
    def mttd_minutes(self) -> float:
        return (self.detection_ts - self.attacker_first_access_ts) / 60

    @property
    def mttr_minutes(self) -> float:
        return (self.containment_complete_ts - self.detection_ts) / 60

    @property
    def total_dwell_minutes(self) -> float:
        return (self.containment_complete_ts - self.attacker_first_access_ts) / 60


class IRPipeline:
    """
    Full incident response pipeline:
    Detection -> Evidence -> Actor-Critic Gate -> Playbook -> Metrics -> LLM Brief
    """

    def __init__(self):
        self.action_log = ActionLog()
        self.metrics_history: List[IRMetrics] = []

    def run(self, incident: Incident, attacker_first_access_ts: float) -> dict:
        """
        Execute the full IR pipeline for one incident.
        Returns complete report dict.
        """
        pipeline_start = time.time()
        report = {
            "incident_id": incident.id,
            "type": incident.type,
            "confidence": incident.confidence,
            "phases": {},
        }

        # ── Phase 1: AI Triage ────────────────────────────────────────────────
        print(f"\n{'='*65}")
        print(f"IR PIPELINE — {incident.id}")
        print(f"{'='*65}")
        print(f"\n[PHASE 1] AI Triage")

        evidence_str = "\n".join(f"  - {e}" for e in incident.evidence)
        triage_prompt = (
            f"Incident {incident.id}: type={incident.type}, host={incident.host}, "
            f"user={incident.user}, confidence={incident.confidence:.0%}.\n"
            f"Evidence:\n{evidence_str}\n\n"
            f"Generate a concise incident brief covering: "
            f"1) Confirmed classification 2) Top 3 immediate priorities "
            f"3) MITRE ATT&CK technique. Under 100 words."
        )
        triage_brief = llm_analyze(triage_prompt, max_tokens=150)
        report["phases"]["triage"] = triage_brief
        print(f"  AI Brief: {triage_brief}")

        # ── Phase 2: Evidence Collection ──────────────────────────────────────
        print(f"\n[PHASE 2] Evidence Collection (pre-containment)")
        forensic_store = ForensicStore(incident.id)
        volatile_collected = 0
        for art_type, size, volatile in [
            ("process_list",   48_200, True),
            ("network_sockets", 12_800, True),
            ("log_file",   1_200_000, False),
        ]:
            forensic_store.collect_artifact(
                art_type, incident.host, f"{art_type} from {incident.host}", size, volatile
            )
            if volatile:
                volatile_collected += 1
        print(f"  Collected {len(forensic_store._artifacts)} artifacts "
              f"({volatile_collected} volatile — sealed before containment)")
        report["phases"]["evidence_artifacts"] = len(forensic_store._artifacts)

        # ── Phase 3: Playbook Execution (with actor-critic gate) ───────────────
        print(f"\n[PHASE 3] Playbook Execution")
        playbook_start = time.time()
        playbook = PLAYBOOKS.get(incident.type, [])
        executed_actions = []
        deferred_actions = []

        for action in playbook:
            # Actor proposes
            proposal = actor_propose(incident, action)
            # Critic evaluates
            verdict  = critic_evaluate(proposal)

            threshold = THRESHOLDS[action.risk]
            if verdict.approved and incident.confidence >= threshold:
                # Log and execute
                rec = self.action_log.record(
                    incident.id, action.name, action.api_endpoint,
                    {"host": incident.host, "user": incident.user},
                    action.rollback,
                )
                executed_actions.append(action.name)
                print(f"  [EXECUTE] {action.name} — critic approved, threshold met")
            else:
                reason = "critic blocked" if not verdict.approved else "below threshold"
                deferred_actions.append(action.name)
                print(f"  [DEFER]   {action.name} — {reason} -> human queue")

        report["phases"]["executed"] = executed_actions
        report["phases"]["deferred"] = deferred_actions
        containment_ts = time.time()

        # ── Phase 4: MTTD/MTTR Measurement ────────────────────────────────────
        print(f"\n[PHASE 4] MTTD/MTTR Metrics")
        metrics = IRMetrics(
            incident_id=incident.id,
            attacker_first_access_ts=attacker_first_access_ts,
            detection_ts=incident.detected_at,
            response_start_ts=pipeline_start,
            containment_complete_ts=containment_ts,
        )
        self.metrics_history.append(metrics)

        mttd_str = f"{metrics.mttd_minutes:.1f} min" if metrics.mttd_minutes < 60 else f"{metrics.mttd_minutes/60:.1f} hr"
        mttr_str = f"{metrics.mttr_minutes*60:.1f} sec" if metrics.mttr_minutes < 1 else f"{metrics.mttr_minutes:.2f} min"
        dwell_str = f"{metrics.total_dwell_minutes:.1f} min" if metrics.total_dwell_minutes < 60 else f"{metrics.total_dwell_minutes/60:.1f} hr"

        print(f"  MTTD  (attacker access -> alert) : {mttd_str}")
        print(f"  MTTR  (alert -> containment)     : {mttr_str}")
        print(f"  Total dwell time                 : {dwell_str}")
        print(f"  Industry baseline MTTD           : 197 days")
        print(f"  Industry baseline MTTR           : 2-8 hours")

        report["metrics"] = {
            "mttd_str": mttd_str,
            "mttr_str": mttr_str,
            "dwell_str": dwell_str,
        }

        # ── Phase 5: LLM Final Brief ───────────────────────────────────────────
        print(f"\n[PHASE 5] LLM Incident Brief (for analyst handoff)")
        brief_prompt = (
            f"Generate a concise incident handoff brief for {incident.id}:\n"
            f"Type: {incident.type} | Host: {incident.host} | User: {incident.user}\n"
            f"Confidence: {incident.confidence:.0%}\n"
            f"Actions executed: {executed_actions}\n"
            f"Actions deferred (need human): {deferred_actions}\n"
            f"MTTD: {mttd_str} | MTTR: {mttr_str}\n"
            f"Evidence: {len(forensic_store._artifacts)} artifacts collected and sealed.\n\n"
            f"Brief should cover: situation, what automated system did, what analyst must decide next. "
            f"Under 120 words."
        )
        final_brief = llm_analyze(brief_prompt, max_tokens=180)
        report["final_brief"] = final_brief
        print(f"  {final_brief}")

        return report


# ── Run the full pipeline ──────────────────────────────────────────────────────
pipeline = IRPipeline()

inc = Incident(
    id="INC-2026-0420",
    type="credential_compromise",
    host="workstation-finance-03",
    user="a.patel",
    confidence=0.92,
    evidence=[
        "Login from 91.108.56.130 (known Tor exit, RU) at 02:11 UTC",
        "Last legitimate login: 10.0.2.45 (London office) 6 hours earlier",
        "MFA challenge declined — user not at desk (HR confirms travel to Singapore)",
        "Accessed payroll-db-01 and exported 220MB — never accessed this host before",
        "Sent 3 emails to external free-mail addresses from a.patel's account",
    ],
    detected_at=time.time() - 45,   # detected 45 seconds ago
)

# Attacker first accessed the account 4 hours before detection
attacker_first_access = inc.detected_at - (4 * 3600)

final_report = pipeline.run(inc, attacker_first_access)

print(f"\n{'='*65}")
print("PIPELINE COMPLETE")
print(f"  Incident   : {final_report['incident_id']}")
print(f"  Executed   : {final_report['phases']['executed']}")
print(f"  Deferred   : {final_report['phases']['deferred']}")
print(f"  MTTD       : {final_report['metrics']['mttd_str']} (vs 197-day industry avg)")
print(f"  MTTR       : {final_report['metrics']['mttr_str']} (vs 2-8 hour industry avg)")
print(f"\nAll deferred actions require analyst approval before execution.")
print("Human-in-the-loop is non-negotiable for medium and high-risk containment.")

## Summary: What You Built

You now have a working **Incident Response Automation** system covering the full NIST IR lifecycle:

| Demo | Component | Key Mechanism | NIST Phase |
|------|-----------|--------------|------------|
| **1. Playbook Executor** | Confidence-gated action runner | Risk-tier thresholds (0.70/0.85/0.95) | Containment |
| **2. Checkpoint + Rollback** | Immutable action log + reversal | Reverse-chronological rollback | Recovery |
| **3. Actor-Critic Gate** | Dual-agent safety architecture | 5-check critic evaluation | All phases |
| **4. Evidence Preservation** | Chain of custody with SHA-256 | Collection precedes containment | Detection/Analysis |
| **5. Full IR Pipeline** | End-to-end orchestration | MTTD/MTTR tracking + LLM brief | All phases |

### The Non-Negotiable Rule

> **Every AI recommendation requires human approval before execution — especially
> for medium-risk and high-risk actions. The actor-critic gate enforces this at the
> architecture level. Never deploy autonomous IR without confidence gating, rollback
> capability, and a human escalation path for blocked actions.**

### Why These Metrics Matter

- **MTTD improvement**: 197 days → 24 hours = 196 fewer days of attacker dwell time per incident
- **MTTR improvement**: 2-8 hours → 1-10 minutes = breach cost reduction (IBM: ~$1M per 30-day improvement)
- **False positive rate**: gates autonomy expansion — earn autonomous execution through demonstrated accuracy

### Production Upgrade Path

```
Simulated API calls          ->  Real EDR/IdP/FW API integrations
In-memory action log         ->  Kafka / append-only database (event sourcing)
SHA-256 content simulation   ->  Real forensic snapshot APIs (Velociraptor, GRR)
Single incident pipeline     ->  Parallel orchestration (Temporal.io / Apache Airflow)
Hardcoded playbooks          ->  YAML/JSON playbook library with version control
claude-haiku-4-5-20251001    ->  claude-sonnet-4-5-20250514 for complex multi-stage IR
```

**Next: Chapter 70 — AI-Powered Threat Detection** — the detection layer that feeds
incidents into this response pipeline. Detection accuracy drives playbook confidence
scores — the two systems are inseparable.

In [ ]:
# ── Production Deployment Checklist + Key Formulas ────────────────────────────
print("CHAPTER 66: INCIDENT RESPONSE AUTOMATION — PRODUCTION CHECKLIST")
print("=" * 65)

checklist = [
    # API Layer
    ("API Layer",        "EDR API: endpoint isolation, process kill, memory snapshot"),
    ("API Layer",        "Firewall API: rule creation, IP block, ACL modification"),
    ("API Layer",        "Identity Provider API: account disable, session revoke, MFA reset"),
    ("API Layer",        "SIEM API: alert creation, evidence query, timeline reconstruction"),
    ("API Layer",        "Ticketing API: incident creation, deferred-action approval workflow"),
    # Playbook Design
    ("Playbook Design",  "Every action has: description, risk tier, rollback procedure, API endpoint"),
    ("Playbook Design",  "Confidence thresholds: low=0.70, medium=0.85, high=0.95"),
    ("Playbook Design",  "Evidence collection always precedes any containment action"),
    ("Playbook Design",  "PLAYBOOK VERSION CONTROLLED — changes require change review"),
    # Safety Architecture
    ("Safety",           "Actor-critic gate: 5 checks before any autonomous execution"),
    ("Safety",           "Critical asset list: production DBs, DCs, payment GW — always defer"),
    ("Safety",           "Rollback required for ALL actions before autonomous deployment"),
    ("Safety",           "Human approval queue: every deferred action with full context"),
    # Metrics
    ("Metrics",          "Baseline MTTD and MTTR before deploying automation"),
    ("Metrics",          "Track false positive rate per incident type and action tier"),
    ("Metrics",          "Autonomy expansion gated by FP rate: low<15%, med<5%, high<2%"),
    ("Metrics",          "Analyst agreement rate: AI recommendation vs analyst decision"),
    # LLM Integration
    ("LLM Integration",  "claude-haiku-4-5-20251001 for triage briefs (fast, cost-effective)"),
    ("LLM Integration",  "claude-sonnet-4-5-20250514 for complex multi-stage IR analysis"),
    ("LLM Integration",  "Always include: evidence list + ATT&CK technique + recommended actions"),
    ("LLM Integration",  "LLM output = recommendation only — analyst decides on execution"),
]

current_cat = ""
for category, item in checklist:
    if category != current_cat:
        print(f"\n[{category}]")
        current_cat = category
    print(f"  + {item}")

print()
print("=" * 65)
print("KEY FORMULAS")
print("=" * 65)
print("Confidence gate : action_executes = (confidence >= THRESHOLDS[risk_tier])")
print("Risk thresholds : low=0.70  |  medium=0.85  |  high=0.95")
print("MTTD            : detection_timestamp - attacker_first_access_timestamp")
print("MTTR            : containment_complete_timestamp - detection_timestamp")
print("FP rate         : false_positive_incidents / total_auto_response_incidents")
print("Rollback order  : reverse(sorted(actions, key=sequence_num))")
print()
print("NIST IR phases covered:")
print("  Preparation   -> playbook library + API layer + confidence thresholds")
print("  Detection     -> LLM triage brief + evidence collection + chain of custody")
print("  Containment   -> confidence-gated playbook + actor-critic safety gate")
print("  Recovery      -> automated rollback + analyst handoff brief")
print("  Post-Incident -> MTTD/MTTR metrics + false positive rate tracking")
print()
print("Chapter 66 complete. Build the pipeline. Gate the confidence.")
print("Every automated action needs a rollback. Human approval is non-negotiable.")